# CMGAN-George Training Notebook

Dieses Notebook startet das Training von `cmgan-george` direkt aus Jupyter.

Vor dem Start bitte sicherstellen:
- `__config__.py` zeigt auf deine lokalen Datenpfade.
- `models/huBERT_wrapper.py` verweist auf das richtige HuBERT-Modell.
- Die gewünschte Hyperparameter-Datei liegt unter `hparams/`.

## Standard-Setup

Die Zellen unten prüfen zuerst den Projektpfad und starten danach `train.py` mit einer Standard-Konfiguration.

In [ ]:
!python -m pip install "pip<24.1" "setuptools<70" wheel "cython<3"
!python -m pip install speechbrain==1.0.2 hyperpyyaml pesq tensorboard onnxruntime torchinfo pyloudnorm glob2 pyroomacoustics einops transformers torchaudio librosa
!python -m pip install git+https://github.com/schmiph2/pysepm.git
!python -m pip install fairseq

In [ ]:
from pathlib import Path
import os
import sys

candidate_roots = [
    Path.cwd(),
    Path.cwd() / 'cmgan-george',
    Path('/workspaces/MetricGAN-KAN-LwF/cmgan-george'),
]
repo_root = next((path for path in candidate_roots if (path / 'train.py').exists()), None)
if repo_root is None:
    raise FileNotFoundError('Konnte cmgan-george/train.py nicht finden. Bitte das Notebook im Workspace öffnen.')

os.chdir(repo_root)
print(f'Repo root: {repo_root}')
print(f'Python: {sys.executable}')

import sys
hparams_file = Path('hparams/hyperparams_chime_bak_ovr_pesq_1.0.yaml')
if not hparams_file.exists():
    raise FileNotFoundError(f'Hyperparameter-Datei fehlt: {hparams_file}')

print(f'Using hparams: {hparams_file}')

In [ ]:
import shutil
import subprocess
import sys


def parse_pyver(ver_str):
    parts = ver_str.strip().split('.')
    return tuple(int(p) for p in parts[:2])


runner = [sys.executable]
pyver = sys.version.split()[0]
conda = shutil.which('conda')

if conda is not None:
    probe = subprocess.run(
        [conda, 'run', '-n', 'CHIME2023', 'python', '-c', 'import sys; print(sys.version.split()[0])'],
        capture_output=True,
        text=True,
    )
    if probe.returncode == 0:
        runner = [conda, 'run', '-n', 'CHIME2023', 'python']
        pyver = probe.stdout.strip()

strict_python_check = False
if parse_pyver(pyver) >= (3, 10):
    msg = (
        f'Aktiver Python ist {pyver}. fairseq in der George-Originalvariante ist oft nur mit 3.8/3.9 stabil. '
        'Ich versuche trotzdem den Run; falls fairseq crasht, brauchst du weiterhin eine 3.8/3.9-Umgebung.'
    )
    if strict_python_check:
        raise RuntimeError(msg)
    print('WARNUNG:', msg)

run_dir = (Path('runs') / hparams_file.stem).resolve()
run_dir.mkdir(parents=True, exist_ok=True)
overrides = [f'output_folder: {run_dir.as_posix()}/']
command = runner + ['train.py', str(hparams_file)] + overrides
print(' '.join(command))
subprocess.run(command, check=True)

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys


def parse_pyver(ver_str):
    parts = ver_str.strip().split('.')
    return tuple(int(p) for p in parts[:2])


runner = [sys.executable]
pyver = sys.version.split()[0]
conda = shutil.which('conda')

if conda is not None:
    probe = subprocess.run(
        [conda, 'run', '-n', 'CHIME2023', 'python', '-c', 'import sys; print(sys.version.split()[0])'],
        capture_output=True,
        text=True,
    )
    if probe.returncode == 0:
        runner = [conda, 'run', '-n', 'CHIME2023', 'python']
        pyver = probe.stdout.strip()

strict_python_check = False
if parse_pyver(pyver) >= (3, 10):
    msg = (
        f'Aktiver Python ist {pyver}. fairseq in der George-Originalvariante ist oft nur mit 3.8/3.9 stabil. '
        'Ich versuche trotzdem den Batch-Run; falls fairseq crasht, brauchst du weiterhin eine 3.8/3.9-Umgebung.'
    )
    if strict_python_check:
        raise RuntimeError(msg)
    print('WARNUNG:', msg)

hparams_dir = Path('hparams')
if not hparams_dir.exists():
    raise FileNotFoundError('Ordner hparams nicht gefunden.')

# Primär: train*.yaml; Fallback: hyperparams*.yaml
hparam_files = sorted(hparams_dir.glob('train*.yaml'))
if not hparam_files:
    hparam_files = sorted(hparams_dir.glob('hyperparams*.yaml'))

if not hparam_files:
    raise RuntimeError('Keine passenden Trainings-YAMLs gefunden (train*.yaml oder hyperparams*.yaml).')

print(f'Gefundene YAMLs: {len(hparam_files)}')
for idx, cfg in enumerate(hparam_files, start=1):
    run_dir = (Path('runs') / cfg.stem).resolve()
    run_dir.mkdir(parents=True, exist_ok=True)
    overrides = [f'output_folder: {run_dir.as_posix()}/']
    cmd = runner + ['train.py', str(cfg)] + overrides
    print(f'[{idx}/{len(hparam_files)}] ' + ' '.join(cmd))
    subprocess.run(cmd, check=True)

print('Alle Trainingsläufe abgeschlossen.')

## Batch-Run über alle Trainings-YAMLs

Diese Zelle läuft automatisch über alle Konfigurationsdateien in `hparams/`.
Priorität: zuerst `train*.yaml`, falls nichts gefunden wird `hyperparams*.yaml`.